# Modelo para la Predicción de Adiciones en Tiempo y Valor

Preguntas de investigación del artículo:
1. ¿Qué tan precisas pueden ser predicciones de las desviacione en tiempo y costo en proyectos de infraestructura?
2. ¿Que tan relevantes pueden ser los diferentes grupos de variables del proyecto: características del proyecto y el proceso de contratación, indicadores de desempeño de la entidad, indicadores de desempeño del contratista, y variablex exogenas?
3. Caules son las principales características del proyecto que determinan las desviaciones
4. Caules son los principales indicadores de la entidad para determinar proyecto que determinan las desviaciones
5. Caules son los principales indicadores del contratista para determinar proyecto que determinan las desviaciones
6. Caules son las principales variables exogenas para determinar proyecto que determinan las desviaciones



In [3]:
# install pycaret in silance
!pip install pycaret --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.7/169.7 kB 5.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 486.1/486.1 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.8/106.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.8/21.8 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.4/85.4 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.2/302.2 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 64.5 MB/s eta 0:00:00
  

In [1]:
%load_ext autoreload
%autoreload 2

In [4]:
# Import data science libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pycaret.classification import *
pd.set_option('display.max_columns', None)

[autoreload of numpy.lib.format failed: Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/IPython/extensions/autoreload.py", line 245, in check
    superreload(m, reload, self.old_objects)
  File "/usr/local/lib/python3.11/dist-packages/IPython/extensions/autoreload.py", line 394, in superreload
    module = reload(module)
             ^^^^^^^^^^^^^^
  File "/usr/lib/python3.11/imp.py", line 315, in reload
    return importlib.reload(module)
           ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.11/importlib/__init__.py", line 169, in reload
    _bootstrap._exec(spec, module)
  File "<frozen importlib._bootstrap>", line 621, in _exec
  File "<frozen importlib._bootstrap_external>", line 940, in exec_module
  File "<frozen importlib._bootstrap>", line 241, in _call_with_frames_removed
  File "/usr/local/lib/python3.11/dist-packages/numpy/lib/format.py", line 166, in <module>
    from numpy.lib.utils import safe_eval, drop_metadata
  File "/usr/loca

AttributeError: module 'pandas.io' has no attribute 'formats'

In [ ]:
#Configurar conexion con google drive y con carpeta de google colab
from google.colab import drive
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/Colab Notebooks/Prediccion de Adiciones Proyectos de Infraestructura')

# Carga de Datos

In [ ]:
df = pd.read_csv('trainData.csv')   # Load the data

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25312 entries, 0 to 25311
Data columns (total 48 columns):
 #   Column                                                        Non-Null Count  Dtype  
---  ------                                                        --------------  -----  
 0   buyerDepartment                                               25312 non-null  object 
 1   buyerLevel                                                    25312 non-null  object 
 2   contractMethodType                                            25312 non-null  object 
 3   buyerRegion                                                   25312 non-null  object 
 4   supplierDepartment                                            25312 non-null  object 
 5   supplierRegion                                                25312 non-null  object 
 6   isBusinessGroup                                               25312 non-null  int64  
 7   isSME                                                         25312

In [ ]:
# Load list of columns
with open('categoricalColumns.txt', 'r') as f:
    categoricalColumns = f.read().splitlines()
with open('numericColumns.txt', 'r') as f:
    numericColumns = f.read().splitlines()
with open('dateColumns.txt', 'r') as f:
    dateColumns = f.read().splitlines()
with open('deviationColumns.txt', 'r') as f:
    deviationColumns = f.read().splitlines()
with open('indicatorsColumns.txt', 'r') as f:
    indicatorsColumns = f.read().splitlines()
with open('terridataIndicators.txt', 'r') as f:
    terridataIndicators = f.read().splitlines()


# Metodología

Entrenar modelos: seleccionar el conjunto del top 5 con el mejor desempeño
Calibrar el top5 de los modelos
Definir n escenarios con diferentes subconjuntos de variables para verificar el aporte de estos nuevos subconjuntos
Comparar el top5 de los modelos con los diferentes escenarios
Realizar un análisis e interpretación de las variables más importantes (Verificar el enfoque de gallego)